In [1]:
!pip install -q unsloth
print("Unsloth installed.")

import unsloth
print(f"Unsloth: {unsloth.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 24.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 88.3 MB/s eta 0:00:00:00:01
  

In [2]:
from unsloth import FastVisionModel
import torch

model, processor = FastVisionModel.from_pretrained(
    "unsloth/llava-v1.6-mistral-7b-hf",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
FastVisionModel.for_inference(model)
print("Model loaded in 4-bit via Unsloth.")
print(f"GPU free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

==((====))==  Unsloth 2026.7.3: Fast Clip patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/687 [00:00<?, ?it/s]

Model loaded in 4-bit via Unsloth.
GPU free: 11.15 GB


In [6]:
from pathlib import Path
import datetime
import cv2

all_videos = list(Path("/kaggle/input").rglob("*.mp4"))
print(f"Found {len(all_videos)} videos:")
for v in all_videos:
    print(f"  {v}")

FRAMES_ROOT = Path("/kaggle/working/frames")
OUTPUT_ROOT = Path("/kaggle/working/llava_outputs")
FRAMES_ROOT.mkdir(exist_ok=True)
OUTPUT_ROOT.mkdir(exist_ok=True)

video_map = {}
for v in all_videos:
    if v.name.startswith("ltx"):
        video_map["ltx"] = v
    elif v.name.startswith("hunyuan"):
        video_map["hunyuan"] = v
    elif v.name.startswith("wan"):
        video_map["wan"] = v

assert len(video_map) == 3, f"Expected 3 videos, found {len(video_map)}: {list(video_map.keys())}"

def extract_frames(video_path, output_folder, num_frames=6):
    output_folder.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    step = (total_frames - 2) / (num_frames - 1)
    frame_indices = [int(1 + i * step) for i in range(num_frames)]
    stem = video_path.stem
    for i, idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            t = idx / fps if fps > 0 else 0
            out = output_folder / f"{stem}_frame{i+1:02d}_t{t:.2f}s.jpg"
            cv2.imwrite(str(out), frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
    cap.release()
    print(f"  {stem}: {num_frames} frames extracted")

for gen, video in video_map.items():
    print(f"From {video.name}:")
    extract_frames(video, FRAMES_ROOT / gen)

print("\nDone.")

Found 3 videos:
  /kaggle/input/datasets/shantanuvedanteog/msc-test-videos/wan_001_20260717_182819.mp4
  /kaggle/input/datasets/shantanuvedanteog/msc-test-videos/hunyuan_001_20260716_193318.mp4
  /kaggle/input/datasets/shantanuvedanteog/msc-test-videos/ltx_001_20260716_133303.mp4
From wan_001_20260717_182819.mp4:
  wan_001_20260717_182819: 6 frames extracted
From hunyuan_001_20260716_193318.mp4:
  hunyuan_001_20260716_193318: 6 frames extracted
From ltx_001_20260716_133303.mp4:
  ltx_001_20260716_133303: 6 frames extracted

Done.


In [7]:
PER_FRAME_PROMPT = """You are a forensic image analyst examining a still frame extracted from a short video. Your task is to look VERY CAREFULLY for visual defects, imperfections, and inconsistencies that indicate the image was synthetically generated by an AI model rather than captured by a real camera.

Modern AI video generators produce subtle but detectable artefacts. Common ones include:
- Skin that looks too smooth, plastic, or waxy (no realistic pores, no fine texture)
- Hair strands that merge together, look painted, or have unnatural clumping
- Eyes with subtly wrong reflections, asymmetric pupils, or slightly-off gaze direction
- Teeth that blend together or have wrong count
- Fingers/hands with wrong number of digits, weird joints, or unnatural bending
- Backgrounds with blurred or nonsensical detail, garbled text on signs
- Lighting inconsistencies (shadows going the wrong way, mismatched light sources)
- Clothing patterns that morph, don't align at seams, or have impossible physics
- Jewellery, glasses, or accessories that distort or merge with skin
- Over-uniform textures on walls, foliage, or fabric

Assume this frame IS AI-generated until proven otherwise. Look actively for at least 2-3 defects. Do not describe the scene at face value — describe what looks WRONG.

Rate the following 5 categories:

1. FACIAL COHERENCE: Look at eyes, teeth, ears, face symmetry, skin texture. Rate 0 (perfect) to 10 (severe defects).

2. TEMPORAL CONSISTENCY: N/A for single frame, rate 0.

3. PHYSICAL PLAUSIBILITY: Look at lighting, shadows, reflections, depth of field. Rate 0 to 10.

4. TEXTURE AND DETAIL: Look at skin pores, hair strands, fabric weave, background detail. Rate 0 to 10.

5. SEMANTIC COHERENCE: Look at hands, fingers, text, object counts, logical scene. Rate 0 to 10.

CRITICAL: A rating of 0 means you looked hard and found ABSOLUTELY nothing wrong — this is very rare for AI-generated content. Most AI frames deserve ratings between 3 and 8 somewhere. If you rate everything 0, you are not looking carefully enough — re-examine the frame.

For each category, provide:
- Severity rating 0-10
- Specific defect you observed (be concrete: name the body part, texture, or object)

Then:
- Likely AI-generated: yes/no/uncertain
- Confidence: integer 0-100 (must be between 0 and 100, never above 100)
- 1-sentence summary of the main defects

Format:

CATEGORY 1 - Facial Coherence: [rating]/10
Observations: [specific defect]

CATEGORY 2 - Temporal Consistency: N/A (single frame)

CATEGORY 3 - Physical Plausibility: [rating]/10
Observations: [specific defect]

CATEGORY 4 - Texture and Detail: [rating]/10
Observations: [specific defect]

CATEGORY 5 - Semantic Coherence: [rating]/10
Observations: [specific defect]

OVERALL:
Likely AI-generated: [yes/no/uncertain]
Confidence: [0-100]%
Summary: [one sentence]"""


print("Prompt v2 loaded — forensic framing with active defect hunting.")

Prompt v2 loaded — forensic framing with active defect hunting.


In [12]:
import re
from statistics import mean

def parse_frame_response(response):
    """Extract ratings and verdict from a single frame's response."""
    ratings = {}
    for cat_num, cat_name in [(1, "Facial Coherence"), (3, "Physical Plausibility"),
                               (4, "Texture and Detail"), (5, "Semantic Coherence")]:
        match = re.search(rf"CATEGORY {cat_num}[^:]*:\s*(\d+)/10", response)
        if match:
            ratings[cat_name] = int(match.group(1))
    
    verdict_match = re.search(r"Likely AI-generated:\s*(\w+)", response, re.IGNORECASE)
    verdict = verdict_match.group(1).lower() if verdict_match else "uncertain"
    
    conf_match = re.search(r"Confidence:\s*(\d+)", response)
    confidence = int(conf_match.group(1)) if conf_match else 0
    confidence = min(confidence, 100)
    
    observations = {}
    for cat_num, cat_name in [(1, "Facial Coherence"), (3, "Physical Plausibility"),
                               (4, "Texture and Detail"), (5, "Semantic Coherence")]:
        match = re.search(rf"CATEGORY {cat_num}[^\n]*\nObservations:\s*([^\n]+)", response)
        if match:
            observations[cat_name] = match.group(1).strip()
    
    return {"ratings": ratings, "verdict": verdict,
            "confidence": confidence, "observations": observations}


def aggregate_video_analysis(frame_dir, generator_name):
    """Analyse all frames individually, aggregate into single video-level verdict."""
    frame_paths = sorted(frame_dir.glob("*.jpg"))
    print(f"Analysing {len(frame_paths)} frames from {frame_dir.name}...")
    
    parsed_frames = []
    for i, fp in enumerate(frame_paths):
        print(f"  Frame {i+1}/{len(frame_paths)}")
        resp = analyze_single_frame(fp)
        parsed_frames.append(parse_frame_response(resp))
    
    # Aggregate: mean rating per category across frames
    categories = ["Facial Coherence", "Physical Plausibility", "Texture and Detail", "Semantic Coherence"]
    mean_ratings = {}
    for cat in categories:
        vals = [f["ratings"].get(cat) for f in parsed_frames if cat in f["ratings"]]
        mean_ratings[cat] = round(mean(vals), 1) if vals else None
    
    # Aggregate verdict: majority vote
    verdicts = [f["verdict"] for f in parsed_frames]
    yes_count = sum(1 for v in verdicts if v == "yes")
    no_count = sum(1 for v in verdicts if v == "no")
    if yes_count > no_count:
        final_verdict = "yes"
    elif no_count > yes_count:
        final_verdict = "no"
    else:
        final_verdict = "uncertain"
    
    # Aggregate confidence: mean of confidences that agreed with majority
    matching = [f["confidence"] for f in parsed_frames if f["verdict"] == final_verdict]
    mean_confidence = round(mean(matching)) if matching else round(mean([f["confidence"] for f in parsed_frames]))
    
    # Collect unique observations per category
    unique_obs = {}
    for cat in categories:
        obs_list = [f["observations"].get(cat, "") for f in parsed_frames if cat in f["observations"]]
        obs_list = [o for o in obs_list if o]
        # Deduplicate while preserving order
        seen = set()
        unique = []
        for o in obs_list:
            key = o.lower()[:50]
            if key not in seen:
                seen.add(key)
                unique.append(o)
        unique_obs[cat] = unique[:3]  # Top 3 unique observations
    
    # Build aggregated video-level response
    summary = f"""CATEGORY 1 - Facial Coherence: {mean_ratings['Facial Coherence']}/10 (mean across {len(frame_paths)} frames)
Observations: {' | '.join(unique_obs['Facial Coherence']) if unique_obs['Facial Coherence'] else 'No specific defects noted.'}

CATEGORY 2 - Temporal Consistency: [Aggregated from per-frame analysis]
Observations: LLaVA-1.6 analysed frames individually as it does not support multi-image sequence input. Temporal consistency inferred from between-frame variation in observations.

CATEGORY 3 - Physical Plausibility: {mean_ratings['Physical Plausibility']}/10 (mean across {len(frame_paths)} frames)
Observations: {' | '.join(unique_obs['Physical Plausibility']) if unique_obs['Physical Plausibility'] else 'No specific defects noted.'}

CATEGORY 4 - Texture and Detail: {mean_ratings['Texture and Detail']}/10 (mean across {len(frame_paths)} frames)
Observations: {' | '.join(unique_obs['Texture and Detail']) if unique_obs['Texture and Detail'] else 'No specific defects noted.'}

CATEGORY 5 - Semantic Coherence: {mean_ratings['Semantic Coherence']}/10 (mean across {len(frame_paths)} frames)
Observations: {' | '.join(unique_obs['Semantic Coherence']) if unique_obs['Semantic Coherence'] else 'No specific defects noted.'}

OVERALL:
Likely AI-generated: {final_verdict}
Confidence: {mean_confidence}%
Summary: Aggregated from {len(frame_paths)} per-frame analyses. Frame-level verdicts: {yes_count} yes, {no_count} no, {len(verdicts) - yes_count - no_count} uncertain. Video-level verdict by majority vote.

--- AGGREGATION METADATA ---
Per-frame verdicts: {verdicts}
Per-frame confidences: {[f['confidence'] for f in parsed_frames]}
Number of frames analysed: {len(frame_paths)}
"""
    return summary


def save_response(response, generator, video_id="001", suffix="v2"):
    header = f"""# Model: unsloth/llava-v1.6-mistral-7b-hf (4-bit via Unsloth)
# Date: {datetime.date.today().isoformat()}
# Input: 6 frames from {generator}_{video_id}.mp4 (analysed individually, aggregated)
# Prompt version: analysis_prompt_{suffix} (forensic reframing, per-frame)
# Access: Kaggle free tier (T4, Unsloth 4-bit quantization)
# Note: LLaVA-1.6 does not support multi-image sequence input; each frame analysed
# individually then aggregated (mean rating per category, majority-vote verdict).
---
{response}
"""
    out_path = OUTPUT_ROOT / f"{generator}_{video_id}_llava_{suffix}.txt"
    out_path.write_text(header)
    print(f"Saved: {out_path}")


# Run all three videos
for generator in ["ltx", "hunyuan", "wan"]:
    print("=" * 60)
    print(f"{generator.upper()} - aggregated per-frame analysis")
    print("=" * 60)
    aggregated = aggregate_video_analysis(FRAMES_ROOT / generator, generator)
    print("\n--- Aggregated Response ---")
    print(aggregated)
    save_response(aggregated, generator)
    print()

LTX - aggregated per-frame analysis
Analysing 6 frames from ltx...
  Frame 1/6
  Frame 2/6
  Frame 3/6
  Frame 4/6
  Frame 5/6
  Frame 6/6

--- Aggregated Response ---
CATEGORY 1 - Facial Coherence: 0/10 (mean across 6 frames)
Observations: The skin texture appears too smooth and lacks realistic pores.

CATEGORY 2 - Temporal Consistency: [Aggregated from per-frame analysis]
Observations: LLaVA-1.6 analysed frames individually as it does not support multi-image sequence input. Temporal consistency inferred from between-frame variation in observations.

CATEGORY 3 - Physical Plausibility: 0/10 (mean across 6 frames)
Observations: The lighting and shadows are inconsistent, with shadows going the wrong way.

CATEGORY 4 - Texture and Detail: 0/10 (mean across 6 frames)
Observations: The hair strands appear to merge together and have an unnatural clumping.

CATEGORY 5 - Semantic Coherence: 0/10 (mean across 6 frames)
Observations: The background is blurred and lacks any nonsensical detail or